In [ ]:
!pip install kagglehub
import kagglehub

path = kagglehub.dataset_download("ektarr/gta-vi-user-reviews")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

files = os.listdir(path)
print('file yang tersedia:', files)

csv_file = os.path.join(path,files[0])
df=pd.read_csv(csv_file)
pd.set_option('display.max_column', None)

In [ ]:
df.isnull().sum()
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df.info()
df.columns.to_list()
display(df.head())

In [ ]:
kata_negatif =['no', 'bad', 'disappointed', 'horrible', 'boring', 'hardly']
kata_positif = ['best', 'enjoy', 'fun', 'well', 'absolute']

def sentimen(teks):
    comment = str(teks).lower()
    if any (kata in comment for kata in kata_positif):
        return 'positif'
    elif any(kata in comment for kata in kata_negatif):
        return 'negatif'
    else:
        return 'netral'
df['review'] = df['review_text'].apply(sentimen)
count= df['review'].value_counts()
print(count)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(data=df, x='review') #countplot itu buat lgsg ngehitung jumlah nya
plt.title('jumlah sentimen berdasarkan review text', fontsize=11)
plt.xlabel('sentimen')
plt.ylabel('jumlah')
plt.tight_layout()
plt.show()

In [ ]:
platform= df['platform'].str.lower().value_counts()
persen = ((platform / len(df))*100).round(2)
print(persen)

plt.figure()
plt.pie(data= df, x= persen, labels= platform.index, autopct='%.2f%%') #autopct harus diawali '% trs .2f itu 2 angka dibelakang koma, trs % itu buat ngasi persen nya, ditutup lagi sm %'
plt.title('presentase penggunaan platform', pad=5)
plt.tight_layout()
plt.show()


In [ ]:
df['published_date'] = pd.to_datetime(df['published_date'])
urutan_tanggal = df['published_date'].sort_values(ascending=True)
print(urutan_tanggal)


In [ ]:
jumlah_skor= df['score'].value_counts()
skor= ((jumlah_skor/len(df))*100).round(2).sort_index()
print(skor)

plt.figure()
plt.bar(skor.index.astype('str'), skor.values) # index itu ambil label skor nya, trs values itu buat ambil tinggi nya, gapake data= karna default nya cuma x,height jd gaperlu manggil data
plt.title('diagram presentase rating')
plt.xlabel('skor')
plt.ylabel('presentase (%)')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.sparse import hstack
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix


def klasifikasi(skor):
    if skor <= 4:
        return 'rendah'
    elif skor <= 7:
        return 'sedang'
    else:
        return 'tinggi'
    
df_bersih = df.dropna(subset=['review_text', 'score', 'platform']).copy() #pake dropna lagi buat jaga" aja, trs subset di isi nama kolom yg mau di dropna, trs copy
y =df['score'].apply(klasifikasi)
print(y.value_counts())

a= TfidfVectorizer()
komen= a.fit_transform(df_bersih['review_text'])

b= OneHotEncoder()
media= b.fit_transform(df_bersih[['platform']])
x= hstack([komen, media]) # hstack itu buat gabungin antara review text sm platform


x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.20,random_state=42, stratify=y) # stratify buat jaga proporsi tiap kelas tetap sama


c= RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
c.fit(x_train,y_train)
y_pred=c.predict(x_test)
print('confusion matrix:\n', confusion_matrix(y_test,y_pred))
print('classification report:\n', classification_report(y_test,y_pred))

kelas rendah
bener nebak kl dia kelas rendah 79%, trs berhasil nebak 94% (sisanya 65 salah nebak)
kelas sedang
bener nebak kl kelas sedang 96%, 3% ketebak sedang (sisanya 97% salah nebak)
kelas tinggi
bener nebak kelas tinggi 88%, 83% berhasil nebak tinggi (17% salah nebak)

confusion matrix, baris itu urutan (rendah, sedang,tinggi)
4892 beneran rendah, 1 rendah dianggap sedang, 334 rendah dianggap tinggi
610 sedang dianggap rendah, 22 beneran sedang, 137 sedang dianggap tinggi
715 tinggi dianggap rendah, 0 tinggi dianggap sedang, 3566 beneran tinggi